<a href="https://colab.research.google.com/github/AnnabelleMcSharry/AI-ML-Car-Data/blob/main/associationrule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
!pip install apyori
from apyori import apriori
import pandas as pd

In [45]:
event = pd.read_csv('drug_adverse_event_dataset.csv')
print('Dimensions of dataset are :', event.shape)
print(event.head())

Dimensions of dataset are : (3500, 10)
   patient_id  age  gender      condition        drug_1        drug_2  \
0       20139   82  Female   Hypertension     Potassium  Atorvastatin   
1       21210   32  Female   Hypertension  Atorvastatin    Phenelzine   
2       21367   70    Male   Hypertension      Warfarin    Fluoxetine   
3       20242   26    Male  Heart Failure      Warfarin    Phenelzine   
4       20514   68  Female       Diabetes    Fluoxetine           NaN   

       drug_3        drug_4  num_medications  adverse_event  
0  Phenelzine           NaN                3              1  
1    Warfarin           NaN                3              0  
2  Phenelzine           NaN                3              1  
3   Metformin  Atorvastatin                4              1  
4         NaN           NaN                1              0  


In [46]:
pd.set_option('display.max_rows', None)  #shows the count of all drug values
print(event["drug_1"].value_counts()) #high number in each category so i do not need to get rid of any drugs that will cause a very high confidence if there is a low number of occurences

drug_1
Metformin       422
Aspirin         401
Fluoxetine      395
Lisinopril      393
Potassium       392
Phenelzine      385
Warfarin        376
Atorvastatin    374
Amoxicillin     362
Name: count, dtype: int64


In [47]:
#put the drugs into a single list for each record so they can be processed
drug_columns = ['drug_1', 'drug_2', 'drug_3', 'drug_4']
drug_list = []

for index, row in event.iterrows():
    drugs_in_row = [str(row[col]) for col in drug_columns if pd.notnull(row[col])]
    if drugs_in_row:
        drug_list.append(drugs_in_row)
    else:
        drug_list.append([]) # Append an empty list if no drugs are found for the row

event['drug_list'] = drug_list

# Drop the original drug columns
event = event.drop(columns=drug_columns)

print(f"Number of drug_list: {len(drug_list)}")
print("First 5 drug_list:")
for t in drug_list[:5]:
    print(t)

print("\nUpdated DataFrame head:")
print(event.head())

Number of drug_list: 3500
First 5 drug_list:
['Potassium', 'Atorvastatin', 'Phenelzine']
['Atorvastatin', 'Phenelzine', 'Warfarin']
['Warfarin', 'Fluoxetine', 'Phenelzine']
['Warfarin', 'Phenelzine', 'Metformin', 'Atorvastatin']
['Fluoxetine']

Updated DataFrame head:
   patient_id  age  gender      condition  num_medications  adverse_event  \
0       20139   82  Female   Hypertension                3              1   
1       21210   32  Female   Hypertension                3              0   
2       21367   70    Male   Hypertension                3              1   
3       20242   26    Male  Heart Failure                4              1   
4       20514   68  Female       Diabetes                1              0   

                                         drug_list  
0            [Potassium, Atorvastatin, Phenelzine]  
1             [Atorvastatin, Phenelzine, Warfarin]  
2               [Warfarin, Fluoxetine, Phenelzine]  
3  [Warfarin, Phenelzine, Metformin, Atorvastatin]  
4  

In [48]:
columns_to_check_duplicates = [col for col in event.columns if col != 'drug_list']
duplicates = event[columns_to_check_duplicates].duplicated()
print(f"Number of duplicate rows (excluding 'drug_list' column): {duplicates.sum()}")
#the duplicated function cannot process a python list so I did not include that but this shows there are no duplicated patients so all of the rows should eb kept

# Check for missing data
#do not want to delete 669 rows of data but can still use the rows if i do not use the conditions column in my rule association
missing_data = event.isnull().sum()
print("\nMissing data in each column:")
print(missing_data)

Number of duplicate rows (excluding 'drug_list' column): 0

Missing data in each column:
patient_id           0
age                  0
gender               0
condition          669
num_medications      0
adverse_event        0
drug_list            0
dtype: int64


In [49]:
# Create the three groups needed for association rule mining

# 1. All patients
All_patients_group = (
    event.groupby("patient_id")["drug_list"]
    .sum()
    .reset_index()
)

# 2. Female patients only
Women_group = (
    event[event["gender"] == "Female"]
    .groupby("patient_id")["drug_list"]
    .sum()
    .reset_index()
)

# 3. Patients over 75 years old
Over_75_group = (
    event[event["age"] > 75]
    .groupby("patient_id")["drug_list"]
    .sum()
    .reset_index()
)

print("All patients group:")
print(All_patients_group.head())

print("\nWomen group:")
print(Women_group.head())

print("\nOver 75 group:")
print(Over_75_group.head())


All patients group:
   patient_id                              drug_list
0       20000                [Lisinopril, Metformin]
1       20001  [Warfarin, Amoxicillin, Atorvastatin]
2       20002              [Amoxicillin, Phenelzine]
3       20003      [Warfarin, Metformin, Fluoxetine]
4       20004               [Lisinopril, Fluoxetine]

Women group:
   patient_id                                     drug_list
0       20003             [Warfarin, Metformin, Fluoxetine]
1       20004                      [Lisinopril, Fluoxetine]
2       20006  [Amoxicillin, Aspirin, Lisinopril, Warfarin]
3       20007                           [Aspirin, Warfarin]
4       20008                         [Aspirin, Phenelzine]

Over 75 group:
   patient_id                  drug_list
0       20014     [Potassium, Metformin]
1       20016                  [Aspirin]
2       20021                [Potassium]
3       20022      [Metformin, Warfarin]
4       20025  [Metformin, Atorvastatin]


In [50]:
All_patients_transactions = All_patients_group['drug_list'].to_list()
Women_transactions = Women_group['drug_list'].to_list()
Over_75_transactions = Over_75_group['drug_list'].to_list()

print(All_patients_transactions)

All_patients_rules = apriori(
    transactions = All_patients_transactions,
    min_support = 3 / len(All_patients_transactions),
    min_confidence = 0.2,
    min_lift = 1.0,
    min_length = 2,
    max_length = 2
)

Women_rules = apriori(
    transactions = Women_transactions,
    min_support = 3 / len(Women_transactions),
    min_confidence = 0.2,
    min_lift = 1.0,
    min_length = 2,
    max_length = 2
)

Over_75_rules = apriori(
    transactions = Over_75_transactions,
    min_support = 3 / len(Over_75_transactions),
    min_confidence = 0.2,
    min_lift = 1.0,
    min_length = 2,
    max_length = 2
)


[['Lisinopril', 'Metformin'], ['Warfarin', 'Amoxicillin', 'Atorvastatin'], ['Amoxicillin', 'Phenelzine'], ['Warfarin', 'Metformin', 'Fluoxetine'], ['Lisinopril', 'Fluoxetine'], ['Fluoxetine'], ['Amoxicillin', 'Aspirin', 'Lisinopril', 'Warfarin'], ['Aspirin', 'Warfarin'], ['Aspirin', 'Phenelzine'], ['Fluoxetine', 'Amoxicillin'], ['Amoxicillin', 'Lisinopril'], ['Amoxicillin', 'Lisinopril', 'Fluoxetine'], ['Fluoxetine'], ['Fluoxetine', 'Lisinopril'], ['Potassium', 'Metformin'], ['Phenelzine', 'Fluoxetine', 'Aspirin'], ['Aspirin'], ['Phenelzine'], ['Amoxicillin', 'Warfarin', 'Fluoxetine'], ['Potassium'], ['Potassium'], ['Potassium'], ['Metformin', 'Warfarin'], ['Fluoxetine'], ['Aspirin', 'Amoxicillin'], ['Metformin', 'Atorvastatin'], ['Lisinopril', 'Amoxicillin', 'Fluoxetine'], ['Aspirin', 'Lisinopril', 'Amoxicillin', 'Warfarin'], ['Lisinopril', 'Warfarin'], ['Warfarin'], ['Potassium', 'Atorvastatin'], ['Lisinopril', 'Atorvastatin', 'Phenelzine', 'Amoxicillin'], ['Fluoxetine', 'Phenelzine'

In [51]:
# Displaying the first results coming directly from the output of the apriori function

All_patients_results = list(All_patients_rules)
Women_results = list(Women_rules)
Over_75_results = list(Over_75_rules)

print(All_patients_results)


[RelationRecord(items=frozenset({'Amoxicillin'}), support=0.27685714285714286, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Amoxicillin'}), confidence=0.27685714285714286, lift=1.0)]), RelationRecord(items=frozenset({'Aspirin'}), support=0.2757142857142857, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Aspirin'}), confidence=0.2757142857142857, lift=1.0)]), RelationRecord(items=frozenset({'Atorvastatin'}), support=0.27914285714285714, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Atorvastatin'}), confidence=0.27914285714285714, lift=1.0)]), RelationRecord(items=frozenset({'Fluoxetine'}), support=0.26857142857142857, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'Fluoxetine'}), confidence=0.26857142857142857, lift=1.0)]), RelationRecord(items=frozenset({'Lisinopril'}), support=0.2737142857142857, ordered_statistics=[OrderedStatistic(items_ba

In [53]:
# Putting the results well organised into a Pandas DataFrame

def inspect(results):
    lhs         = [tuple(result[2][0][0])[0] for result in results]
    rhs         = [tuple(result[2][0][1])[0] for result in results]
    supports    = [result[1] for result in results]
    confidences = [result[2][0][2] for result in results]
    lifts       = [result[2][0][3] for result in results]
    return list(zip(lhs, rhs, supports, confidences, lifts))


In [54]:
All_patients_results_DataFrame = pd.DataFrame(
    inspect(All_patients_results),
    columns = ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift']
)

Women_results_DataFrame = pd.DataFrame(
    inspect(Women_results),
    columns = ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift']
)

Over_75_results_DataFrame = pd.DataFrame(
    inspect(Over_75_results),
    columns = ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift']
)


IndexError: tuple index out of range